# 1 Database-related functions
These functions are for establishing a connection to the database, reading tables from it and sending tables to it. They will be re-used in larger functions for fetching and injecting the weather forecasts and arriving flights into the database.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lat_lon_parser import parse
import requests
from dotenv import load_dotenv #to hide passwords
load_dotenv()
import os
from pytz import timezone
from datetime import date, timedelta, datetime

def get_passwords_and_api_keys():
    #Assigning the api keys from the env file to variables
    api_key_airports = os.getenv("API_KEY_FLIGHTS_MINE") #for airports
    api_key_flights = os.getenv("API_KEY_FLIGHTS_WBS") #for arriving flights
    api_key_weather = os.getenv("API_KEY_WEATHER") #for weather forecasts
    #Assigning the database password, from the env file
    db_password = os.getenv("MY_SQL_PASSWORD")
    return db_password, api_key_airports, api_key_flights, api_key_weather

In [27]:
def create_connection_string(db_password):
    schema = "weather_flights_db"
    host = "207.175.55.125" #for cloud
    user = "root"
    password = db_password
    port = 3306
    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [8]:
def fetch_df_from_sql(connection_string, df_name): #df name in quotes!
    return pd.read_sql(df_name, con=connection_string)

In [10]:
def send_df_to_sql(connection_string, df_name, table_name_in_db): #table_name in quotation marks when using the function!
    df_name.to_sql(table_name_in_db,
                if_exists='append',
                con=connection_string,
                index=False)

# 2 One-time functions
Run only when a new city is added to the database (e.g. when the company expands to it)

## 2.1 Get city coordinates based on city names
This is done through webscraping: We input a list of cities, get out a data frame with the cities' coordinates, and send this data frame to the database. The coordinates are webscrped from the English version of Wikipedia.

In [11]:
def get_city_data(list_of_city_names_in_eng):
    city_data = []

    for city in list_of_city_names_in_eng:
        url = f"https://www.wikipedia.org/wiki/{city}"
        headers = {'User-Agent': 'Chrome/134.0.0.0'}

        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, 'html.parser')

        # extract the relevant information
        country = city_soup.find(class_="infobox-data").get_text()
        city_latitude = city_soup.find(class_="latitude").get_text()
        city_longitude = city_soup.find(class_="longitude").get_text()

        # for each city we append a dictionary of values to the list
        city_data.append({"city_name": city,
                        "country": country,
                        "latitude": parse(city_latitude), # latitude in decimal format, using the parse function
                        "longitude": parse(city_longitude)
                        })

    cities_df = pd.DataFrame(city_data)
    return cities_df

## 2.2 Get nearby airports based on city coordinates
This is done through an API call: we supply the city coordinates that we retrieved in the previous step, and get the airports in the 50 km radius.

For this function, we need the API key and the data frame that has city_ids as well as their latitudes and longitudes.

The function returns 2 data frames: one with just city-ids and associated airport codes in icao, and the other one with extended airport information (name, municipality, icao and iata codes). Both data frames should be injected into the data base using send_to_sql() defined in the previous section.

In [37]:
def get_city_airports(api_key_airports, city_df):
    city_id_list = city_df.loc[:,"city_id"].to_list()
    lat_list = city_df.loc[:,"latitude"].to_list()
    lon_list = city_df.loc[:,"longitude"].to_list()

    headers = {
        "x-rapidapi-key": api_key_airports,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }
    all_airports = []

    for city_id, lat, lon in zip(city_id_list, lat_list, lon_list):
        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {"withFlightInfoOnly":"True","radiusKm":"50","limit":"10","lat":lat,"lon":lon}
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200: #.status_code is an attribute of the reponse object (it's its own type)
                    data = response.json() #returns response in the json format
                    items = data.get('items', []) #returns a list of dictionaries because under the key "item" a list of dict was stored (or an empty list)
                    airports = pd.json_normalize(items) #this returns a data frame, based on the list of dictionaries
                    airports["city_id"] = city_id
                    all_airports.append(airports) #now this is a list of data frames!!

    all_airports      

    all_airports_df = pd.concat(all_airports, ignore_index=True) #this stacks the data frames from the list into 1 df, and we need to ignore the indeces from the original dfs, because they will be repetitive
    airports_df = airports_df = all_airports_df.drop(columns="city_id")
    cities_airports_df = all_airports_df[["city_id", "icao"]]

    return airports_df, cities_airports_df

# 3 Recurring functions

Run periodically to update the information about the weather in the cities of interest and about the arriving flights.

## 3.1 Get 5-day forecast for the cities

In [13]:
def fetch_weather_data(api_key_weather, city_df):
    
    city_id_list = city_df.loc[:,"city_id"].to_list()
    latitude_list = city_df.loc[:,"latitude"].to_list()
    longitude_list = city_df.loc[:,"longitude"].to_list()

    combined_weather_json = {}

    for city_id, lat, lon in zip(city_id_list, latitude_list, longitude_list):
        weather_request = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&units=metric&appid={api_key_weather}")
        weather_json = weather_request.json()
        combined_weather_json[city_id] = weather_json

    temp = {}
    feels_like = {}
    temp_min = {}
    temp_max = {}
    rain_main = {}
    rain_description = {}
    perc_clouds = {}
    wind_speed = {}
    wind_gust = {}
    prob_rain = {}
    date_time = {}

    for city_id in city_id_list:
        temp[city_id] = []
        feels_like[city_id] = []
        temp_min[city_id] = []
        temp_max[city_id] = []
        rain_main[city_id] = []
        rain_description[city_id] = []
        perc_clouds[city_id] = []
        wind_speed[city_id] = []
        wind_gust[city_id] = []
        prob_rain[city_id] = []
        date_time[city_id] = []
        for i in range(40):
            temp[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp"])
            feels_like[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["feels_like"])
            temp_min[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp_min"])
            temp_max[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp_max"])
            rain_main[city_id].append(combined_weather_json[city_id]["list"][i]["weather"][0]["main"])
            rain_description[city_id].append(combined_weather_json[city_id]["list"][i]["weather"][0]["description"])
            perc_clouds[city_id].append(combined_weather_json[city_id]["list"][i]["clouds"]["all"])
            wind_speed[city_id].append(combined_weather_json[city_id]["list"][i]["wind"]["speed"])
            wind_gust[city_id].append(combined_weather_json[city_id]["list"][i]["wind"]["gust"])
            prob_rain[city_id].append(combined_weather_json[city_id]["list"][i]["pop"])
            date_time[city_id].append(combined_weather_json[city_id]["list"][i]["dt_txt"])

    berlin_timezone = timezone('Europe/Berlin')
    rows = []
    for city_id in city_id_list:
        for i in range(40):
            rows.append({
                "city_id": city_id,
                "temp": temp[city_id][i],
                "feels_like": feels_like[city_id][i],
                "temp_min": temp_min[city_id][i],
                "temp_max": temp_max[city_id][i],
                "sky": rain_main[city_id][i],
                "sky_description": rain_description[city_id][i],
                "perc_clouds": perc_clouds[city_id][i],
                "wind_speed": wind_speed[city_id][i],
                "wind_gust": wind_gust[city_id][i],
                "prob_rain": prob_rain[city_id][i],
                "date_time": date_time[city_id][i],
                "retrieval_time": datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")
                })
    
    return pd.DataFrame(rows) 

## 3.2 Get tomorrow's arriving flights for the airports near the cities

In [ ]:
def get_airport_arrivals_for_tomorrow(api_key_flights, cities_airports_df):
    airport_codes = cities_airports_df.loc[:, "icao"].to_list()
    start_times = ["00:00", "12:00"]
    end_times = ["11:59", "23:59"]
    berlin_timezone = timezone('Europe/Berlin')
    tomorrow = datetime.now(berlin_timezone).date() + timedelta(days=1)
    tomorrow_str = tomorrow.strftime("%Y-%m-%d")
    arrivals_per_airport = {}

    headers = {
        "x-rapidapi-key": api_key_flights,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }       

    for airport_code in airport_codes:
        arrival_times_sch_local = [] #empty list for each airport, arrival times scheduled in local time
        arrival_times_revised_local = [] #arrival times revised in local time
        arrival_times_sch_utc = []
        arrival_times_revised_utc = []
        dep_times_sch_utc = [] #dep times scheduled in utc
        dep_times_revised_utc = [] 
        dep_airports_icao = [] #Departure airport in icao
        dep_airports_name = [] #Departure airport name - to see if scooters are popular there too
        dep_airports_country = [] #Departure airport country
        airlines = []
        
        for start_time, end_time in zip(start_times, end_times):
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport_code}/{tomorrow_str}T{start_time.split(":")[0]}%3A{start_time.split(":")[1]}/{tomorrow_str}T{end_time.split(":")[0]}%3A{end_time.split(":")[1]}"

            querystring = {"withCargo":"false",
                    "withCodeshared":"true",
                    "direction":"Arrival",
                    "withCancelled":"false",
                    "withLeg":"true",
                    "withPrivate":"false",
                    "withLocation":"false"}

            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 200:             
                    flights_json = response.json()              
                    for i in range(len(flights_json["arrivals"])):
                        arrival_time_sch_local = flights_json.get("arrivals", [])[i].get("arrival", {}).get("scheduledTime", {}).get("local")
                        if arrival_time_sch_local is not None:
                            arrival_time_sch_local = arrival_time_sch_local.split("+")[0] + ":00"
                        arrival_times_sch_local.append(arrival_time_sch_local) 
                        
                        arrival_time_revised_local = flights_json.get("arrivals", [])[i].get("arrival", {}).get("revisedTime", {}).get("local")
                        if arrival_time_revised_local is not None:
                            arrival_time_revised_local = arrival_time_revised_local.split("+")[0] + ":00"
                        arrival_times_revised_local.append(arrival_time_revised_local)

                        arrival_time_sch_utc = flights_json.get("arrivals", [])[i].get("arrival", {}).get("scheduledTime", {}).get("utc")
                        if arrival_time_sch_utc is not None:
                            arrival_time_sch_utc = arrival_time_sch_utc.rstrip("Z")+":00"
                        arrival_times_sch_utc.append(arrival_time_sch_utc) 

                        arrival_time_revised_utc = flights_json.get("arrivals", [])[i].get("arrival", {}).get("revisedTime", {}).get("utc")
                        if arrival_time_revised_utc is not None:
                            arrival_time_revised_utc = arrival_time_revised_utc.rstrip("Z")+":00"
                        arrival_times_revised_utc.append(arrival_time_revised_utc) 

                        dep_time_sch_utc = flights_json.get("arrivals", [])[i].get("departure", {}).get("scheduledTime", {}).get("utc")
                        if dep_time_sch_utc is not None:
                            dep_time_sch_utc = dep_time_sch_utc.rstrip("Z")+":00"
                        dep_times_sch_utc.append(dep_time_sch_utc) 

                        dep_time_revised_utc = flights_json.get("arrivals", [])[i].get("departure", {}).get("revisedTime", {}).get("utc")
                        if dep_time_revised_utc is not None:
                            dep_time_revised_utc = dep_time_revised_utc.rstrip("Z")+":00"
                        dep_times_revised_utc.append(dep_time_revised_utc) 

                        dep_airport_icao = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("icao")
                        dep_airports_icao.append(dep_airport_icao)

                        dep_airport_name = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("name")
                        dep_airports_name.append(dep_airport_name)  

                        dep_airport_country = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("countryCode")
                        dep_airports_country.append(dep_airport_country) 

                        airline = flights_json.get("arrivals", [])[i].get("airline", {}).get("name")
                        airlines.append(airline)
                
        arrivals_per_airport[airport_code] = {"arrival_time_sch_local": arrival_times_sch_local,
                                        "arrival_time_revised_local": arrival_times_revised_local,
                                        "arrival_time_sch_utc": arrival_times_sch_utc,
                                        "arrival_time_revised_utc": arrival_times_revised_utc,
                                        "dep_time_sch_utc": dep_times_sch_utc,
                                        "dep_time_revised_utc": dep_times_revised_utc,
                                        "dep_airport_icao": dep_airports_icao,
                                        "dep_airport_name": dep_airports_name,
                                        "dep_airport_country": dep_airports_country, 
                                        "airline": airlines
                                        }
        
    rows = []
    berlin_timezone = timezone('Europe/Berlin')
    
    for airport_code, arrival_data in arrivals_per_airport.items():
        for i in range(len(list(arrival_data.values())[0])):
            row = {
                "airport_code": airport_code,
                "retrieval_time": datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")
                    }  
            for data_name, data_list in arrival_data.items():
                row[data_name] = data_list[i]
            rows.append(row)
    #rows      
    arrivals_df = pd.DataFrame(rows)   
    
    #converting time columns to the datetime format
    datetime_cols = [
        "arrival_time_sch_local",
        "arrival_time_revised_local",
        "arrival_time_sch_utc",
        "arrival_time_revised_utc",
        "dep_time_sch_utc",
        "dep_time_revised_utc"
        ]
    
    for col in datetime_cols:
        arrivals_df[col] = pd.to_datetime(arrivals_df[col])
    
    arrivals_df["flight_length_hrs"] = round(
    (arrivals_df["arrival_time_sch_utc"] - arrivals_df["dep_time_sch_utc"])
        .dt.total_seconds() / 3600,
        2
                                        )
    return arrivals_df    
    

# 4 Functions for Google Cloud

These functions are desgigned to run automatically on Google Cloud - they fetch and inject the necessary data on their own, and they just need to be triggered by a scheduler.

## 4.1 For getting 5-day weather forecasts for cities

In [ ]:
import pandas as pd
import requests
import requests
from dotenv import load_dotenv #to hide passwords
load_dotenv()
import os
from pytz import timezone
from datetime import datetime

def get_weather_for_cities():
    db_password, api_key_weather = get_passwords_and_api_keys()
    connection_string = create_connection_string(db_password)
    cities_df = fetch_df_from_sql(connection_string, "cities") #cities table name hardcoded!
    weather_df = fetch_weather_data(api_key_weather, cities_df)
    send_df_to_sql(connection_string, weather_df, "weather_forecasts") #weather_forecasts table name hardcoded


def get_passwords_and_api_keys():
    api_key_weather = os.getenv("API_KEY_WEATHER") #for weather forecasts
    #Assigning the database password, from the env file
    db_password = os.getenv("MY_SQL_PASSWORD")
    return db_password, api_key_weather #,api_key_airports, api_key_flights,

def create_connection_string(db_password):
    schema = "weather_flights_db"
    host = "207.175.55.125" #for cloud
    user = "root"
    password = db_password
    port = 3306
    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

def fetch_df_from_sql(connection_string, df_name): #df name in quotes!
    return pd.read_sql(df_name, con=connection_string)

def fetch_weather_data(api_key_weather, city_df):
    city_id_list = city_df.loc[:,"city_id"].to_list()
    latitude_list = city_df.loc[:,"latitude"].to_list()
    longitude_list = city_df.loc[:,"longitude"].to_list()

    combined_weather_json = {}

    for city_id, lat, lon in zip(city_id_list, latitude_list, longitude_list):
        weather_request = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&units=metric&appid={api_key_weather}")
        weather_json = weather_request.json()
        combined_weather_json[city_id] = weather_json

    temp = {}
    feels_like = {}
    temp_min = {}
    temp_max = {}
    rain_main = {}
    rain_description = {}
    perc_clouds = {}
    wind_speed = {}
    wind_gust = {}
    prob_rain = {}
    date_time = {}

    for city_id in city_id_list:
        temp[city_id] = []
        feels_like[city_id] = []
        temp_min[city_id] = []
        temp_max[city_id] = []
        rain_main[city_id] = []
        rain_description[city_id] = []
        perc_clouds[city_id] = []
        wind_speed[city_id] = []
        wind_gust[city_id] = []
        prob_rain[city_id] = []
        date_time[city_id] = []
        for i in range(40):
            temp[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp"])
            feels_like[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["feels_like"])
            temp_min[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp_min"])
            temp_max[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp_max"])
            rain_main[city_id].append(combined_weather_json[city_id]["list"][i]["weather"][0]["main"])
            rain_description[city_id].append(combined_weather_json[city_id]["list"][i]["weather"][0]["description"])
            perc_clouds[city_id].append(combined_weather_json[city_id]["list"][i]["clouds"]["all"])
            wind_speed[city_id].append(combined_weather_json[city_id]["list"][i]["wind"]["speed"])
            wind_gust[city_id].append(combined_weather_json[city_id]["list"][i]["wind"]["gust"])
            prob_rain[city_id].append(combined_weather_json[city_id]["list"][i]["pop"])
            date_time[city_id].append(combined_weather_json[city_id]["list"][i]["dt_txt"])

    berlin_timezone = timezone('Europe/Berlin')
    rows = []
    for city_id in city_id_list:
        for i in range(40):
            rows.append({
                "city_id": city_id,
                "temp": temp[city_id][i],
                "feels_like": feels_like[city_id][i],
                "temp_min": temp_min[city_id][i],
                "temp_max": temp_max[city_id][i],
                "sky": rain_main[city_id][i],
                "sky_description": rain_description[city_id][i],
                "perc_clouds": perc_clouds[city_id][i],
                "wind_speed": wind_speed[city_id][i],
                "wind_gust": wind_gust[city_id][i],
                "prob_rain": prob_rain[city_id][i],
                "date_time": date_time[city_id][i],
                "retrieval_time": datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")
                })
    
    new_weather_df = pd.DataFrame(rows) 

    #The cloud scheduler will run every 5th day of the month, so on the 1st, 6th, 11th, 16th, 21st, 26th, 31st.
    #Not every 5 days since you started the scheduler
    #So it will run on the 31st and on the 1st, creating duplicated weather predictions
    #To avoid this, we need to pull the existing weather predictions, compare them to the new ones, 
    # and only send to sql the non-overlapping predictions

    old_weather_df = fetch_df_from_sql(connection_string, "weather_forecasts")
    new_weather_multiindex = new_weather_df.set_index(["city_id", "date_time"]).index
    old_weather_multiindex = old_weather_df.set_index(["city_id", "date_time"]).index
    matches = new_weather_multiindex.isin(old_weather_multiindex)
    new_weather_df = new_weather_df[~matches].copy()
    return new_weather_df

def send_df_to_sql(connection_string, df_name, table_name_in_db): #table_name in quotation marks when using the function!
    df_name.to_sql(table_name_in_db,
                if_exists='append',
                con=connection_string,
                index=False)

## 4.2 For getting tomorrow's arriving flights for cities' airports

In [ ]:
def get_arriving_flights_for_cities():
    db_password, api_key_flights = get_passwords_and_api_keys()
    connection_string = create_connection_string(db_password)
    cities_airports_df = fetch_df_from_sql(connection_string, "cities_airports") #cities_airports table name hardcoded!
    arrivals_df = get_airport_arrivals_for_tomorrow(api_key_flights, cities_airports_df)
    send_df_to_sql(connection_string, arrivals_df, "arriving_flights") #arriving_flights table name hardcoded


def get_passwords_and_api_keys():
    api_key_flights = os.getenv("API_KEY_FLIGHTS_WBS") #for arriving flights
    #api_key_weather = os.getenv("API_KEY_WEATHER") #for weather forecasts
    #Assigning the database password, from the env file
    db_password = os.getenv("MY_SQL_PASSWORD")
    return db_password, api_key_flights #, api_key_airports #  api_key_weather 

def create_connection_string(db_password):
    schema = "weather_flights_db" #db name hardcoded
    host = "207.175.55.125" # instance hardcoded, this is the one for cloud
    user = "root"
    password = db_password
    port = 3306
    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

def fetch_df_from_sql(connection_string, df_name): #df name in quotes!
    return pd.read_sql(df_name, con=connection_string)

def get_airport_arrivals_for_tomorrow(api_key_flights, cities_airports_df):
    airport_codes = cities_airports_df.loc[:, "icao"].to_list()
    start_times = ["00:00", "12:00"]
    end_times = ["11:59", "23:59"]
    berlin_timezone = timezone('Europe/Berlin')
    tomorrow = datetime.now(berlin_timezone).date() + timedelta(days=1)
    tomorrow_str = tomorrow.strftime("%Y-%m-%d")
    arrivals_per_airport = {}

    headers = {
        "x-rapidapi-key": api_key_flights,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }       

    for airport_code in airport_codes:
        arrival_times_sch_local = [] #empty list for each airport, arrival times scheduled in local time
        arrival_times_revised_local = [] #arrival times revised in local time
        arrival_times_sch_utc = []
        arrival_times_revised_utc = []
        dep_times_sch_utc = [] #dep times scheduled in utc
        dep_times_revised_utc = [] 
        dep_airports_icao = [] #Departure airport in icao
        dep_airports_name = [] #Departure airport name - to see if scooters are popular there too
        dep_airports_country = [] #Departure airport country
        airlines = []
        
        for start_time, end_time in zip(start_times, end_times):
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport_code}/{tomorrow_str}T{start_time.split(":")[0]}%3A{start_time.split(":")[1]}/{tomorrow_str}T{end_time.split(":")[0]}%3A{end_time.split(":")[1]}"

            querystring = {"withCargo":"false",
                    "withCodeshared":"true",
                    "direction":"Arrival",
                    "withCancelled":"false",
                    "withLeg":"true",
                    "withPrivate":"false",
                    "withLocation":"false"}

            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 200:             
                    flights_json = response.json()              
                    for i in range(len(flights_json["arrivals"])):
                        arrival_time_sch_local = flights_json.get("arrivals", [])[i].get("arrival", {}).get("scheduledTime", {}).get("local")
                        if arrival_time_sch_local is not None:
                            arrival_time_sch_local = arrival_time_sch_local.split("+")[0] + ":00"
                        arrival_times_sch_local.append(arrival_time_sch_local) 
                        
                        arrival_time_revised_local = flights_json.get("arrivals", [])[i].get("arrival", {}).get("revisedTime", {}).get("local")
                        if arrival_time_revised_local is not None:
                            arrival_time_revised_local = arrival_time_revised_local.split("+")[0] + ":00"
                        arrival_times_revised_local.append(arrival_time_revised_local)

                        arrival_time_sch_utc = flights_json.get("arrivals", [])[i].get("arrival", {}).get("scheduledTime", {}).get("utc")
                        if arrival_time_sch_utc is not None:
                            arrival_time_sch_utc = arrival_time_sch_utc.rstrip("Z")+":00"
                        arrival_times_sch_utc.append(arrival_time_sch_utc) 

                        arrival_time_revised_utc = flights_json.get("arrivals", [])[i].get("arrival", {}).get("revisedTime", {}).get("utc")
                        if arrival_time_revised_utc is not None:
                            arrival_time_revised_utc = arrival_time_revised_utc.rstrip("Z")+":00"
                        arrival_times_revised_utc.append(arrival_time_revised_utc) 

                        dep_time_sch_utc = flights_json.get("arrivals", [])[i].get("departure", {}).get("scheduledTime", {}).get("utc")
                        if dep_time_sch_utc is not None:
                            dep_time_sch_utc = dep_time_sch_utc.rstrip("Z")+":00"
                        dep_times_sch_utc.append(dep_time_sch_utc) 

                        dep_time_revised_utc = flights_json.get("arrivals", [])[i].get("departure", {}).get("revisedTime", {}).get("utc")
                        if dep_time_revised_utc is not None:
                            dep_time_revised_utc = dep_time_revised_utc.rstrip("Z")+":00"
                        dep_times_revised_utc.append(dep_time_revised_utc) 

                        dep_airport_icao = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("icao")
                        dep_airports_icao.append(dep_airport_icao)

                        dep_airport_name = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("name")
                        dep_airports_name.append(dep_airport_name)  

                        dep_airport_country = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("countryCode")
                        dep_airports_country.append(dep_airport_country) 

                        airline = flights_json.get("arrivals", [])[i].get("airline", {}).get("name")
                        airlines.append(airline)
                
        arrivals_per_airport[airport_code] = {"arrival_time_sch_local": arrival_times_sch_local,
                                        "arrival_time_revised_local": arrival_times_revised_local,
                                        "arrival_time_sch_utc": arrival_times_sch_utc,
                                        "arrival_time_revised_utc": arrival_times_revised_utc,
                                        "dep_time_sch_utc": dep_times_sch_utc,
                                        "dep_time_revised_utc": dep_times_revised_utc,
                                        "dep_airport_icao": dep_airports_icao,
                                        "dep_airport_name": dep_airports_name,
                                        "dep_airport_country": dep_airports_country, 
                                        "airline": airlines
                                        }
        
    rows = []
    berlin_timezone = timezone('Europe/Berlin')
    
    for airport_code, arrival_data in arrivals_per_airport.items():
        for i in range(len(list(arrival_data.values())[0])):
            row = {
                "icao": airport_code,
                "retrieval_time": datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")
                    }  
            for data_name, data_list in arrival_data.items():
                row[data_name] = data_list[i]
            rows.append(row)
    #rows      
    arrivals_df = pd.DataFrame(rows)   
    
    #converting time columns to the datetime format
    datetime_cols = [
        "arrival_time_sch_local",
        "arrival_time_revised_local",
        "arrival_time_sch_utc",
        "arrival_time_revised_utc",
        "dep_time_sch_utc",
        "dep_time_revised_utc"
        ]
    
    for col in datetime_cols:
        arrivals_df[col] = pd.to_datetime(arrivals_df[col])
    
    arrivals_df["flight_length_hrs"] = round(
    (arrivals_df["arrival_time_sch_utc"] - arrivals_df["dep_time_sch_utc"])
        .dt.total_seconds() / 3600,
        2
                                        )
    return arrivals_df    

def send_df_to_sql(connection_string, df_name, table_name_in_db): #table_name in quotation marks when using the function!
    df_name.to_sql(table_name_in_db,
                if_exists='append',
                con=connection_string,
                index=False)    

## 4.3 How the functions are written in cloud source code

### 4.3.1 Weather function, main.py

In [ ]:
import pysqlite3
import sys
sys.modules["sqlite3"] = pysqlite3
import functions_framework

import pandas as pd
import requests
import sqlalchemy
import requests
from dotenv import load_dotenv #to hide passwords
load_dotenv()
import os
from pytz import timezone
from datetime import date, timedelta, datetime

@functions_framework.http
def get_weather_for_cities(request):
    db_password, api_key_weather = get_passwords_and_api_keys()
    connection_string = create_connection_string(db_password)
    cities_df = fetch_df_from_sql(connection_string, "cities") #cities table name hardcoded!
    weather_df = fetch_weather_data(api_key_weather, cities_df)
    send_df_to_sql(connection_string, weather_df, "weather_forecasts") #weather_forecasts table name hardcoded
    return "Weather data successfully added to database"

def get_passwords_and_api_keys():
    api_key_weather = os.getenv("API_KEY_WEATHER")
    db_password = os.getenv("MY_SQL_PASSWORD")
    return db_password, api_key_weather

def create_connection_string(db_password):
    schema = "weather_flights_db" #name of database
    host = "207.175.55.125" # the instance for cloud
    user = "root"
    password = db_password
    port = 3306
    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

def fetch_df_from_sql(connection_string, df_name): #df name in quotes!
    return pd.read_sql(df_name, con=connection_string)

def fetch_weather_data(api_key_weather, city_df):
    city_id_list = city_df.loc[:,"city_id"].to_list()
    latitude_list = city_df.loc[:,"latitude"].to_list()
    longitude_list = city_df.loc[:,"longitude"].to_list()

    combined_weather_json = {}

    for city_id, lat, lon in zip(city_id_list, latitude_list, longitude_list):
        weather_request = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&units=metric&appid={api_key_weather}")
        weather_json = weather_request.json()
        combined_weather_json[city_id] = weather_json

    temp = {}
    feels_like = {}
    temp_min = {}
    temp_max = {}
    rain_main = {}
    rain_description = {}
    perc_clouds = {}
    wind_speed = {}
    wind_gust = {}
    prob_rain = {}
    date_time = {}

    for city_id in city_id_list:
        temp[city_id] = []
        feels_like[city_id] = []
        temp_min[city_id] = []
        temp_max[city_id] = []
        rain_main[city_id] = []
        rain_description[city_id] = []
        perc_clouds[city_id] = []
        wind_speed[city_id] = []
        wind_gust[city_id] = []
        prob_rain[city_id] = []
        date_time[city_id] = []
        for i in range(40):
            temp[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp"])
            feels_like[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["feels_like"])
            temp_min[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp_min"])
            temp_max[city_id].append(combined_weather_json[city_id]["list"][i]["main"]["temp_max"])
            rain_main[city_id].append(combined_weather_json[city_id]["list"][i]["weather"][0]["main"])
            rain_description[city_id].append(combined_weather_json[city_id]["list"][i]["weather"][0]["description"])
            perc_clouds[city_id].append(combined_weather_json[city_id]["list"][i]["clouds"]["all"])
            wind_speed[city_id].append(combined_weather_json[city_id]["list"][i]["wind"]["speed"])
            wind_gust[city_id].append(combined_weather_json[city_id]["list"][i]["wind"]["gust"])
            prob_rain[city_id].append(combined_weather_json[city_id]["list"][i]["pop"])
            date_time[city_id].append(combined_weather_json[city_id]["list"][i]["dt_txt"])

    berlin_timezone = timezone('Europe/Berlin')
    rows = []
    for city_id in city_id_list:
        for i in range(40):
            rows.append({
                "city_id": city_id,
                "temp": temp[city_id][i],
                "feels_like": feels_like[city_id][i],
                "temp_min": temp_min[city_id][i],
                "temp_max": temp_max[city_id][i],
                "sky": rain_main[city_id][i],
                "sky_description": rain_description[city_id][i],
                "perc_clouds": perc_clouds[city_id][i],
                "wind_speed": wind_speed[city_id][i],
                "wind_gust": wind_gust[city_id][i],
                "prob_rain": prob_rain[city_id][i],
                "date_time": date_time[city_id][i],
                "retrieval_time": datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")
                })
    
    new_weather_df = pd.DataFrame(rows) 
    #The cloud scheduler will run every 5th day of the month, so on the 1st, 6th, 11th, 16th, 21st, 26th, 31st.
    #Not every 5 days since you started the scheduler
    #So it will run on the 31st and on the 1st, creating duplicated weather predictions
    #To avoid this, we need to pull the existing weather predictions, compare them to the new ones, 
    # and only send to sql the non-overlapping predictions

    old_weather_df = fetch_df_from_sql(connection_string, "weather_forecasts")
    new_weather_multiindex = new_weather_df.set_index(["city_id", "date_time"]).index
    old_weather_multiindex = old_weather_df.set_index(["city_id", "date_time"]).index
    matches = new_weather_multiindex.isin(old_weather_multiindex)
    new_weather_df = new_weather_df[~matches].copy()
    return new_weather_df

def send_df_to_sql(connection_string, df_name, table_name_in_db): #table_name in quotation marks when using the function!
    df_name.to_sql(table_name_in_db,
                if_exists='append',
                con=connection_string,
                index=False)


#### Weather and arriving flights function, requirements.txt

functions-framework==3.*
SQLAlchemy
PyMySQL
pandas
pysqlite3-binary
requests
python-dotenv
pytz

## 4.3.2 Arriving flights function, main.py

In [ ]:
import pysqlite3
import sys
sys.modules["sqlite3"] = pysqlite3
import functions_framework

import pandas as pd
import requests
import sqlalchemy
import requests
from dotenv import load_dotenv #to hide passwords
load_dotenv()
import os
from pytz import timezone
from datetime import date, timedelta, datetime

@functions_framework.http
def get_arriving_flights_for_cities(request):
    db_password, api_key_flights = get_passwords_and_api_keys()
    connection_string = create_connection_string(db_password)
    cities_airports_df = fetch_df_from_sql(connection_string, "cities_airports") #cities_airports table name hardcoded!
    arrivals_df = get_airport_arrivals_for_tomorrow(api_key_flights, cities_airports_df)
    send_df_to_sql(connection_string, arrivals_df, "arriving_flights") #arriving_flights table name hardcoded
    return "Arriving flights data successfully added to database"

def get_passwords_and_api_keys():
    api_key_flights = os.getenv("API_KEY_FLIGHTS_WBS") #for arriving flights
    db_password = os.getenv("MY_SQL_PASSWORD")
    return db_password, api_key_flights

def create_connection_string(db_password):
    schema = "weather_flights_db" #db name hardcoded
    host = "207.175.55.125" # instance hardcoded, this is the one for cloud
    user = "root"
    password = db_password
    port = 3306
    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

def fetch_df_from_sql(connection_string, df_name): #df name in quotes!
    return pd.read_sql(df_name, con=connection_string)

def get_airport_arrivals_for_tomorrow(api_key_flights, cities_airports_df):
    airport_codes = cities_airports_df.loc[:, "icao"].to_list()
    start_times = ["00:00", "12:00"]
    end_times = ["11:59", "23:59"]
    berlin_timezone = timezone('Europe/Berlin')
    tomorrow = datetime.now(berlin_timezone).date() + timedelta(days=1)
    tomorrow_str = tomorrow.strftime("%Y-%m-%d")
    arrivals_per_airport = {}

    headers = {
        "x-rapidapi-key": api_key_flights,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }       

    for airport_code in airport_codes:
        arrival_times_sch_local = [] #empty list for each airport, arrival times scheduled in local time
        arrival_times_revised_local = [] #arrival times revised in local time
        arrival_times_sch_utc = []
        arrival_times_revised_utc = []
        dep_times_sch_utc = [] #dep times scheduled in utc
        dep_times_revised_utc = [] 
        dep_airports_icao = [] #Departure airport in icao
        dep_airports_name = [] #Departure airport name - to see if scooters are popular there too
        dep_airports_country = [] #Departure airport country
        airlines = []
        
        for start_time, end_time in zip(start_times, end_times):
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport_code}/{tomorrow_str}T{start_time.split(":")[0]}%3A{start_time.split(":")[1]}/{tomorrow_str}T{end_time.split(":")[0]}%3A{end_time.split(":")[1]}"

            querystring = {"withCargo":"false",
                    "withCodeshared":"true",
                    "direction":"Arrival",
                    "withCancelled":"false",
                    "withLeg":"true",
                    "withPrivate":"false",
                    "withLocation":"false"}

            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 200:             
                    flights_json = response.json()              
                    for i in range(len(flights_json["arrivals"])):
                        arrival_time_sch_local = flights_json.get("arrivals", [])[i].get("arrival", {}).get("scheduledTime", {}).get("local")
                        if arrival_time_sch_local is not None:
                            arrival_time_sch_local = arrival_time_sch_local.split("+")[0] + ":00"
                        arrival_times_sch_local.append(arrival_time_sch_local) 
                        
                        arrival_time_revised_local = flights_json.get("arrivals", [])[i].get("arrival", {}).get("revisedTime", {}).get("local")
                        if arrival_time_revised_local is not None:
                            arrival_time_revised_local = arrival_time_revised_local.split("+")[0] + ":00"
                        arrival_times_revised_local.append(arrival_time_revised_local)

                        arrival_time_sch_utc = flights_json.get("arrivals", [])[i].get("arrival", {}).get("scheduledTime", {}).get("utc")
                        if arrival_time_sch_utc is not None:
                            arrival_time_sch_utc = arrival_time_sch_utc.rstrip("Z")+":00"
                        arrival_times_sch_utc.append(arrival_time_sch_utc) 

                        arrival_time_revised_utc = flights_json.get("arrivals", [])[i].get("arrival", {}).get("revisedTime", {}).get("utc")
                        if arrival_time_revised_utc is not None:
                            arrival_time_revised_utc = arrival_time_revised_utc.rstrip("Z")+":00"
                        arrival_times_revised_utc.append(arrival_time_revised_utc) 

                        dep_time_sch_utc = flights_json.get("arrivals", [])[i].get("departure", {}).get("scheduledTime", {}).get("utc")
                        if dep_time_sch_utc is not None:
                            dep_time_sch_utc = dep_time_sch_utc.rstrip("Z")+":00"
                        dep_times_sch_utc.append(dep_time_sch_utc) 

                        dep_time_revised_utc = flights_json.get("arrivals", [])[i].get("departure", {}).get("revisedTime", {}).get("utc")
                        if dep_time_revised_utc is not None:
                            dep_time_revised_utc = dep_time_revised_utc.rstrip("Z")+":00"
                        dep_times_revised_utc.append(dep_time_revised_utc) 

                        dep_airport_icao = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("icao")
                        dep_airports_icao.append(dep_airport_icao)

                        dep_airport_name = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("name")
                        dep_airports_name.append(dep_airport_name)  

                        dep_airport_country = flights_json.get("arrivals", [])[i].get("departure", {}).get("airport", {}).get("countryCode")
                        dep_airports_country.append(dep_airport_country) 

                        airline = flights_json.get("arrivals", [])[i].get("airline", {}).get("name")
                        airlines.append(airline)
                
        arrivals_per_airport[airport_code] = {"arrival_time_sch_local": arrival_times_sch_local,
                                        "arrival_time_revised_local": arrival_times_revised_local,
                                        "arrival_time_sch_utc": arrival_times_sch_utc,
                                        "arrival_time_revised_utc": arrival_times_revised_utc,
                                        "dep_time_sch_utc": dep_times_sch_utc,
                                        "dep_time_revised_utc": dep_times_revised_utc,
                                        "dep_airport_icao": dep_airports_icao,
                                        "dep_airport_name": dep_airports_name,
                                        "dep_airport_country": dep_airports_country, 
                                        "airline": airlines
                                        }
        
    rows = []
    berlin_timezone = timezone('Europe/Berlin')
    
    for airport_code, arrival_data in arrivals_per_airport.items():
        for i in range(len(list(arrival_data.values())[0])):
            row = {
                "icao": airport_code,
                "retrieval_time": datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")
                    }  
            for data_name, data_list in arrival_data.items():
                row[data_name] = data_list[i]
            rows.append(row)
    #rows      
    arrivals_df = pd.DataFrame(rows)   
    
    #converting time columns to the datetime format
    datetime_cols = [
        "arrival_time_sch_local",
        "arrival_time_revised_local",
        "arrival_time_sch_utc",
        "arrival_time_revised_utc",
        "dep_time_sch_utc",
        "dep_time_revised_utc"
        ]
    
    for col in datetime_cols:
        arrivals_df[col] = pd.to_datetime(arrivals_df[col])
    
    arrivals_df["flight_length_hrs"] = round(
    (arrivals_df["arrival_time_sch_utc"] - arrivals_df["dep_time_sch_utc"])
        .dt.total_seconds() / 3600,
        2
                                        )
    return arrivals_df    

def send_df_to_sql(connection_string, df_name, table_name_in_db): #table_name in quotation marks when using the function!
    df_name.to_sql(table_name_in_db,
                if_exists='append',
                con=connection_string,
                index=False)    

## 4.3.3 Weather and arriving flights functions, requirements.txt

functions-framework==3.*                    
SQLAlchemy                   
PyMySQL                   
pandas                   
pysqlite3-binary                   
requests                   
python-dotenv                   
pytz                   